In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from getpass import getpass
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

In [5]:
import pandas as pd

In [6]:
DB_NAME = "skku_f3_project"

In [8]:
DB_USER = "root"
DB_PASSWORD = getpass("MySQL 비밀번호: ")
DB_HOST = "localhost"
DB_PORT = 3306 

MySQL 비밀번호:  ········


In [9]:
url = URL.create(
    drivername="mysql+pymysql",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(url)

In [10]:
from sqlalchemy import text

with engine.connect() as conn:
    db = conn.execute(text("SELECT DATABASE();")).scalar()
    
print("연결된 DB:", db)

연결된 DB: skku_f3_project


In [11]:
from sqlalchemy import inspect

inspector = inspect(engine)
table_names = inspector.get_table_names()

print("전체 테이블 수:", len(table_names))

for table in table_names:
    print(table)

전체 테이블 수: 17
accounts
card_payments
cards
customer_debt_summary
customer_financial_profiles
customer_monthly_cashflow
customer_real_estate
customer_stock_lots
customer_stock_positions
customers
devices
fraud_cases
investment_accounts
loans
real_assets
self_employed_businesses
transactions


In [12]:
financial_profiles = pd.read_sql(
    "SELECT * FROM customer_financial_profiles",
    engine
)

financial_profiles.head()

,customer_id,name,birth_date,age,age_band,gender,region,household_size,employment_type,primary_monthly_income,...,calibration_household_income_median,calibration_consumption_monthly,calibration_financial_asset_median,calibration_home_ownership_rate,spouse_present,num_children,school_age_children,monthly_private_education_cost,base_monthly_consumption_budget,education_burden_ratio
0,37,김현우,1991-03-14,35,30대,M,경기도,1,SME,3274858.04,...,61750000,2824000,83600000,0.360,0,0,0,0.0,1975596.36,0.0
1,72,윤윤서,2000-06-19,26,20대,M,전라북도,1,SELF_EMPLOYED,5966624.16,...,40470000,2824000,50000000,0.094,0,0,0,0.0,3214007.67,0.0
2,78,최지우,1997-02-26,29,20대,M,경기도,1,TEMP_DAILY,3789988.59,...,40470000,2824000,50000000,0.094,0,0,0,0.0,2364101.73,0.0
3,86,서지호,1988-03-22,38,30대,M,서울특별시,1,SELF_EMPLOYED,7728467.75,...,61750000,2824000,83600000,0.360,0,0,0,0.0,4634238.87,0.0
4,95,조서연,1995-08-17,31,30대,F,경기도,1,SME,2347367.24,...,61750000,2824000,83600000,0.360,0,0,0,0.0,2856450.71,0.0


In [13]:
financial_profiles[
    ["customer_id", "age", "spouse_present"]
].head()

,customer_id,age,spouse_present
0,37,35,0
1,72,26,0
2,78,29,0
3,86,38,0
4,95,31,0


In [14]:
financial_profiles["age_analysis"] = financial_profiles["age"] + 1

In [15]:
target_before_exclude = financial_profiles[
    (financial_profiles["age_analysis"].between(26, 39)) &
    (financial_profiles["spouse_present"] == False)
].copy()

print(
    "제외 전 고객 수:",
    target_before_exclude["customer_id"].nunique()
)

제외 전 고객 수: 7320


In [16]:
print("전체 고객 수:",
      financial_profiles["customer_id"].nunique())

print("원본 age 26~39세:",
      financial_profiles[
          financial_profiles["age"].between(26, 39)
      ]["customer_id"].nunique())

print("age+1 기준 26~39세:",
      financial_profiles[
          (financial_profiles["age"] + 1).between(26, 39)
      ]["customer_id"].nunique())

print("원본 age 26~39세 + 배우자 없음:",
      financial_profiles[
          (financial_profiles["age"].between(26, 39)) &
          (financial_profiles["spouse_present"] == False)
      ]["customer_id"].nunique())

print("age+1 기준 26~39세 + 배우자 없음:",
      financial_profiles[
          ((financial_profiles["age"] + 1).between(26, 39)) &
          (financial_profiles["spouse_present"] == False)
      ]["customer_id"].nunique())

전체 고객 수: 7747
원본 age 26~39세: 7744
age+1 기준 26~39세: 7320
원본 age 26~39세 + 배우자 없음: 7744
age+1 기준 26~39세 + 배우자 없음: 7320


In [17]:
financial_profiles[
    ~financial_profiles["age"].between(26, 39)
][["customer_id", "age", "spouse_present"]]

,customer_id,age,spouse_present
2987,32706,25,0
4639,50837,25,0
7305,79836,25,0


In [18]:
# 제외할 고객
exclude_ids = [32706, 50837, 79836]

# 최종 분석 대상 고객
target_customers = financial_profiles[
    (financial_profiles["age"].between(26, 39)) &
    (financial_profiles["spouse_present"] == False) &
    (~financial_profiles["customer_id"].isin(exclude_ids))
].copy()

print("최종 고객 수:",
      target_customers["customer_id"].nunique())

최종 고객 수: 7744


In [19]:
target_customers["age_analysis"] = target_customers["age"] + 1

In [20]:
cashflow = pd.read_sql(
    "SELECT * FROM customer_monthly_cashflow",
    engine
)

investment_accounts = pd.read_sql(
    "SELECT * FROM investment_accounts",
    engine
)

stock_lots = pd.read_sql(
    "SELECT * FROM customer_stock_lots",
    engine
)

stock_positions = pd.read_sql(
    "SELECT * FROM customer_stock_positions",
    engine
)

In [21]:
print("cashflow:", cashflow.shape)
print("investment_accounts:", investment_accounts.shape)
print("stock_lots:", stock_lots.shape)
print("stock_positions:", stock_positions.shape)

cashflow: (139446, 18)
investment_accounts: (2578, 9)
stock_lots: (8138, 10)
stock_positions: (8138, 16)


In [22]:
print("cashflow 고객 수:",
      cashflow["customer_id"].nunique())

print("investment_accounts 고객 수:",
      investment_accounts["customer_id"].nunique())

print("stock_lots 고객 수:",
      stock_lots["customer_id"].nunique())

print("stock_positions 고객 수:",
      stock_positions["customer_id"].nunique())

cashflow 고객 수: 7747
investment_accounts 고객 수: 2578
stock_lots 고객 수: 2578
stock_positions 고객 수: 2578


In [23]:
# 분석 대상 고객 ID
target_ids = target_customers["customer_id"]

# 7,744명 기준으로 필터링
cashflow_target = cashflow[
    cashflow["customer_id"].isin(target_ids)
].copy()

investment_accounts_target = investment_accounts[
    investment_accounts["customer_id"].isin(target_ids)
].copy()

stock_lots_target = stock_lots[
    stock_lots["customer_id"].isin(target_ids)
].copy()

stock_positions_target = stock_positions[
    stock_positions["customer_id"].isin(target_ids)
].copy()

In [24]:
print("=== 필터링 후 데이터 ===")

print(
    "cashflow:",
    cashflow_target.shape,
    "/ 고객 수:",
    cashflow_target["customer_id"].nunique()
)

print(
    "investment_accounts:",
    investment_accounts_target.shape,
    "/ 고객 수:",
    investment_accounts_target["customer_id"].nunique()
)

print(
    "stock_lots:",
    stock_lots_target.shape,
    "/ 고객 수:",
    stock_lots_target["customer_id"].nunique()
)

print(
    "stock_positions:",
    stock_positions_target.shape,
    "/ 고객 수:",
    stock_positions_target["customer_id"].nunique()
)

=== 필터링 후 데이터 ===
cashflow: (139392, 18) / 고객 수: 7744
investment_accounts: (2578, 9) / 고객 수: 2578
stock_lots: (8138, 10) / 고객 수: 2578
stock_positions: (8138, 16) / 고객 수: 2578


In [25]:
# financial_profiles 기준 주식 투자자 수
print(
    "is_stock_investor = 1 고객 수:",
    target_customers.loc[
        target_customers["is_stock_investor"] == 1,
        "customer_id"
    ].nunique()
)

# 투자계좌 보유 고객 수
print(
    "investment_accounts 고객 수:",
    investment_accounts_target["customer_id"].nunique()
)

is_stock_investor = 1 고객 수: 2578
investment_accounts 고객 수: 2578


In [26]:
profile_investors = set(
    target_customers.loc[
        target_customers["is_stock_investor"] == 1,
        "customer_id"
    ]
)

account_investors = set(
    investment_accounts_target["customer_id"]
)

print("프로필에만 있는 고객:", len(profile_investors - account_investors))
print("투자계좌에만 있는 고객:", len(account_investors - profile_investors))
print("완전히 동일:", profile_investors == account_investors)

프로필에만 있는 고객: 0
투자계좌에만 있는 고객: 0
완전히 동일: True


In [27]:
debt_summary = pd.read_sql(
    "SELECT * FROM customer_debt_summary",
    engine
)

In [28]:
debt_summary_target = debt_summary[
    debt_summary["customer_id"].isin(target_ids)
].copy()

In [29]:
print(
    "financial_profiles:",
    target_customers.shape,
    "/ 고객 수:",
    target_customers["customer_id"].nunique()
)

print(
    "debt_summary:",
    debt_summary_target.shape,
    "/ 고객 수:",
    debt_summary_target["customer_id"].nunique()
)

financial_profiles: (7744, 39) / 고객 수: 7744
debt_summary: (7744, 23) / 고객 수: 7744


In [30]:
print("=== 투자 파트 분석 데이터 준비 상태 ===")

print(
    "1. financial_profiles:",
    target_customers.shape,
    "/ 고객 수:", target_customers["customer_id"].nunique()
)

print(
    "2. cashflow:",
    cashflow_target.shape,
    "/ 고객 수:", cashflow_target["customer_id"].nunique()
)

print(
    "3. investment_accounts:",
    investment_accounts_target.shape,
    "/ 고객 수:", investment_accounts_target["customer_id"].nunique()
)

print(
    "4. stock_lots:",
    stock_lots_target.shape,
    "/ 고객 수:", stock_lots_target["customer_id"].nunique()
)

print(
    "5. stock_positions:",
    stock_positions_target.shape,
    "/ 고객 수:", stock_positions_target["customer_id"].nunique()
)

print(
    "6. debt_summary:",
    debt_summary_target.shape,
    "/ 고객 수:", debt_summary_target["customer_id"].nunique()
)

=== 투자 파트 분석 데이터 준비 상태 ===
1. financial_profiles: (7744, 39) / 고객 수: 7744
2. cashflow: (139392, 18) / 고객 수: 7744
3. investment_accounts: (2578, 9) / 고객 수: 2578
4. stock_lots: (8138, 10) / 고객 수: 2578
5. stock_positions: (8138, 16) / 고객 수: 2578
6. debt_summary: (7744, 23) / 고객 수: 7744


In [31]:
# customer_financial_profiles 구조 확인
target_customers.info()

<class 'pandas.DataFrame'>
Index: 7744 entries, 0 to 7746
Data columns (total 39 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   customer_id                          7744 non-null   int64  
 1   name                                 7744 non-null   str    
 2   birth_date                           7744 non-null   object 
 3   age                                  7744 non-null   int64  
 4   age_band                             7744 non-null   str    
 5   gender                               7744 non-null   str    
 6   region                               7744 non-null   str    
 7   household_size                       7744 non-null   int64  
 8   employment_type                      7744 non-null   str    
 9   primary_monthly_income               7744 non-null   float64
 10  primary_annual_income                7744 non-null   float64
 11  household_annual_income              7744 non-

In [32]:
for i, col in enumerate(target_customers.columns, 1):
    print(f"{i:2}. {col}")

 1. customer_id
 2. name
 3. birth_date
 4. age
 5. age_band
 6. gender
 7. region
 8. household_size
 9. employment_type
10. primary_monthly_income
11. primary_annual_income
12. household_annual_income
13. disposable_annual_income
14. disposable_ratio
15. monthly_consumption_budget
16. monthly_card_budget
17. financial_assets_estimated
18. savings_propensity
19. investment_propensity
20. stock_investor_probability
21. is_stock_investor
22. investment_risk_profile
23. home_ownership_probability
24. is_home_owner
25. credit_score
26. risk_segment
27. income_band
28. calibration_wage_median_monthly
29. calibration_household_income_median
30. calibration_consumption_monthly
31. calibration_financial_asset_median
32. calibration_home_ownership_rate
33. spouse_present
34. num_children
35. school_age_children
36. monthly_private_education_cost
37. base_monthly_consumption_budget
38. education_burden_ratio
39. age_analysis


In [33]:
investment_eda = target_customers[
    [
        "investment_propensity",
        "stock_investor_probability",
        "is_stock_investor",
        "investment_risk_profile"
    ]
]

investment_eda.head()

,investment_propensity,stock_investor_probability,is_stock_investor,investment_risk_profile
0,0.5968,0.3915,0,BALANCED
1,0.5024,0.2514,0,BALANCED
2,0.4697,0.2328,1,BALANCED
3,0.6538,0.4705,0,BALANCED
4,0.7134,0.4500,0,AGGRESSIVE


In [34]:
investment_eda[
    ["investment_propensity", "stock_investor_probability"]
].describe().T

,count,mean,std,min,25%,50%,75%,max
investment_propensity,7744.0,0.549409,0.108652,0.1968,0.469700,0.5489,0.6306,0.9384
stock_investor_probability,7744.0,0.328869,0.098916,0.1589,0.234875,0.3517,0.4199,0.5569


# investment_propensity — 전반적인 투자 성향
즉 고객들의 투자 성향 점수는 대체로 0.55 부근에 위치하고 있음. 평균과 중앙값이 거의 같기 때문에 적어도 기초통계상으로는 한쪽으로 크게 치우쳐 있다고 보기는 어려움. 다만 정확한 분포 형태는 히스토그램을 봐야 판단할 수 있음.

# stock_investor_probability — 주식 투자 확률


In [35]:
print("=== 주식 투자 여부 ===")
print(
    investment_eda["is_stock_investor"]
    .value_counts()
    .sort_index()
)

print("\n=== 투자 위험 성향 ===")
print(
    investment_eda["investment_risk_profile"]
    .value_counts()
)

=== 주식 투자 여부 ===
is_stock_investor
0    5166
1    2578
Name: count, dtype: int64

=== 투자 위험 성향 ===
investment_risk_profile
BALANCED        6478
AGGRESSIVE       924
CONSERVATIVE     342
Name: count, dtype: int64


In [36]:
risk_by_investor = pd.crosstab(
    target_customers["investment_risk_profile"],
    target_customers["is_stock_investor"],
    margins=True
)

risk_by_investor


#비투자자 5,166명에게도 위험성향이 모두 존재한다는 것이야. 
#즉 investment_risk_profile은 “현재 주식에 투자하고 있는 사람에게만 부여된 값”이 아니라 전체 고객의 투자 위험 선호 성향을 나타내는 프로필 변수

is_stock_investor,0,1,All
investment_risk_profile,,,
AGGRESSIVE,522,402,924
BALANCED,4378,2100,6478
CONSERVATIVE,266,76,342
All,5166,2578,7744


In [37]:
risk_pct = pd.crosstab(
    target_customers["investment_risk_profile"],
    target_customers["is_stock_investor"],
    normalize="columns"
) * 100

risk_pct.round(2)

#주식 투자자 집단에서 AGGRESSIVE 비율이 상대적으로 높고 CONSERVATIVE는 비투자자 5.15% → 투자자 2.95%로, 주식 투자자 집단에서 약 2.20%p 낮음

is_stock_investor,0,1
investment_risk_profile,,
AGGRESSIVE,10.10,15.59
BALANCED,84.75,81.46
CONSERVATIVE,5.15,2.95


# 위험성향 관련 EDA 정리
투자 위험성향과 주식 투자 여부 비교: 전체 고객 중 BALANCED 성향이 가장 높은 비중을 차지했다. 주식 투자자 집단에서는 AGGRESSIVE 비율이 15.59%로 비주식 투자자(10.10%)보다 높았으며, CONSERVATIVE 비율은 주식 투자자 2.95%로 비주식 투자자(5.15%)보다 낮았다. 따라서 위험 선호도가 높은 고객이 주식투자자 집단에서 상대적으로 높은 비중을 보이는 경향이 관찰되었다.

# 테이블 정리
investment_risk_profile
→ 나는 어느 정도 위험을 감수하려는 사람인가? (성향)

stock_lots / investment_transfer
→ 실제로 어떻게 투자했는가? (행동)

stock_positions
→ 실제로 어떤 위험의 자산을 보유하는가? (결과)

In [38]:
target_customers.groupby("is_stock_investor")[
    ["investment_propensity", "stock_investor_probability"]
].agg(["count", "mean", "median", "std", "min", "max"]).round(3)

investment_propensity                                     \
                                  count   mean median    std    min    max   
is_stock_investor                                                            
0                                  5166  0.537  0.533  0.108  0.197  0.874   
1                                  2578  0.574  0.580  0.107  0.254  0.938   

                  stock_investor_probability                              \
                                       count   mean median    std    min   
is_stock_investor                                                          
0                                       5166  0.314  0.269  0.098  0.159   
1                                       2578  0.359  0.403  0.094  0.162   

                          
                     max  
is_stock_investor         
0                  0.557  
1                  0.557

# 정리
investment_propensity를 보면 투자자는 평균 0.574, 비투자자는 0.537이야. 중앙값도 투자자 0.580 > 0.533으로 동일한 방향이야. 즉 실제 주식 투자자 집단에서 전반적인 투자 성향 점수가 상대적으로 높은 경향이 나타나.
stock_investor_probability도 투자자 평균 0.359 > 비투자자 0.314야. 특히 중앙값은 투자자 0.403, 비투자자 0.269로 차이가 더 크게 나타났어.
다만 여기서 중요한 게 있어. 이 결과를 보고 **“이 점수 때문에 투자자가 됐다”**라고 하면 안 돼. 현재는 두 집단의 기술통계를 비교한 것이므로 투자자 집단에서 점수가 상대적으로 높게 나타났다 정도로 해석하는 게 정확해.
그리고 예전에 네가 헷갈렸던 부분도 이제 명확하게 정리할 수 있어.
stock_investor_probability가 높다는 것은 “투자하는 사람 중에서 주식에 투자할 확률”이라는 의미라기보다, 전체 고객에게 부여된 주식투자 가능성 점수로 보는 게 적절해. 실제 투자 여부는 별도의 is_stock_investor로 구분돼 있어.     F3 EER diagram 2

# [customer_financial_profiles EDA 결과]

- 전체 분석 대상 7,744명 중 주식 투자자는 2,578명(33.3%),
  비주식 투자자는 5,166명(66.7%)이다.

- 투자 위험성향은 전체적으로 BALANCED가 가장 높은 비중을 차지한다.

- 주식 투자자 집단은 비투자자 집단보다 AGGRESSIVE 비율이 높고,
  CONSERVATIVE 비율이 낮은 경향을 보였다.

- 주식 투자자의 investment_propensity 평균은 0.574로
  비투자자(0.537)보다 높았다.

- stock_investor_probability 역시 투자자 평균 0.359로
  비투자자(0.314)보다 높았다.

→ 실제 주식 투자자 집단에서 투자 적극성 및 위험 선호와 관련된
  프로필 점수가 상대적으로 높은 경향이 관찰된다.

# customer_monthly_cashflow EDA

In [39]:
#1단계 — 데이터 구조 확인
cashflow_target.info()

cashflow_target.head()

<class 'pandas.DataFrame'>
Index: 139392 entries, 0 to 139445
Data columns (total 18 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   customer_id                     139392 non-null  int64  
 1   month                           139392 non-null  str    
 2   income_inflow                   139392 non-null  float64
 3   card_spend                      139392 non-null  float64
 4   education_card_spend            139392 non-null  float64
 5   private_education_spend_target  139392 non-null  float64
 6   education_noncard_estimated     139392 non-null  float64
 7   non_card_consumption            139392 non-null  float64
 8   debt_service                    139392 non-null  float64
 9   monthly_debt_service_ratio      139392 non-null  float64
 10  housing_education_fixed_burden  139392 non-null  float64
 11  housing_education_burden_ratio  139392 non-null  float64
 12  savings_transfer                

,customer_id,month,income_inflow,card_spend,education_card_spend,private_education_spend_target,education_noncard_estimated,non_card_consumption,debt_service,monthly_debt_service_ratio,housing_education_fixed_burden,housing_education_burden_ratio,savings_transfer,investment_transfer,fraud_loss_account_transfer,total_outflow,net_cashflow,month_end_balance
0,37,2025-04,3328083.98,1317600.0,0.0,0.0,0.0,598807.65,931731.72,0.241444,931731.72,0.300205,115142.59,0.0,0.0,2963281.96,364802.02,12779414.45
1,37,2025-05,3240473.76,1423000.0,0.0,0.0,0.0,493407.65,931731.72,0.241444,931731.72,0.300205,128379.42,0.0,0.0,2976518.78,263954.97,13043369.42
2,37,2025-06,3390984.61,586000.0,0.0,0.0,0.0,1330407.65,931731.72,0.241444,931731.72,0.300205,106249.90,0.0,0.0,2954389.27,436595.34,13479964.76
3,37,2025-07,3348883.74,1424800.0,0.0,0.0,0.0,491607.65,931731.72,0.241444,931731.72,0.300205,103964.40,0.0,0.0,2952103.77,396779.97,13876744.73
4,37,2025-08,3302977.30,1269800.0,177600.0,0.0,0.0,646607.65,931731.72,0.241444,931731.72,0.300205,110770.23,0.0,0.0,2958909.59,344067.71,14220812.44


In [40]:
cashflow_cols = [
    "customer_id",
    "month",
    "income_inflow",
    "savings_transfer",
    "investment_transfer"
]

cashflow_target[cashflow_cols].head(10)

,customer_id,month,income_inflow,savings_transfer,investment_transfer
0,37,2025-04,3328083.98,115142.59,0.0
1,37,2025-05,3240473.76,128379.42,0.0
2,37,2025-06,3390984.61,106249.90,0.0
3,37,2025-07,3348883.74,103964.40,0.0
4,37,2025-08,3302977.30,110770.23,0.0
5,37,2025-09,3382424.52,106019.72,0.0
6,37,2025-10,3174828.90,99447.55,0.0
7,37,2025-11,3237108.52,134776.88,0.0
8,37,2025-12,3320530.86,101488.58,0.0
9,37,2026-01,3210373.86,97762.03,0.0


In [41]:
# 2단계 — 기간 확인
print("최초 월:", cashflow_target["month"].min())
print("최근 월:", cashflow_target["month"].max())
print("전체 월 수:", cashflow_target["month"].nunique())

sorted(cashflow_target["month"].unique())

최초 월: 2025-04
최근 월: 2026-09
전체 월 수: 18


['2025-04',
 '2025-05',
 '2025-06',
 '2025-07',
 '2025-08',
 '2025-09',
 '2025-10',
 '2025-11',
 '2025-12',
 '2026-01',
 '2026-02',
 '2026-03',
 '2026-04',
 '2026-05',
 '2026-06',
 '2026-07',
 '2026-08',
 '2026-09']

In [42]:
months_per_customer = (
    cashflow_target
    .groupby("customer_id")["month"]
    .nunique()
)

months_per_customer.describe()

count    7744.0
mean       18.0
std         0.0
min        18.0
25%        18.0
50%        18.0
75%        18.0
max        18.0
Name: month, dtype: float64

In [43]:
months_per_customer.value_counts().sort_index()

month
18    7744
Name: count, dtype: int64

In [44]:
print("최초 월:", cashflow_target["month"].min())
print("최근 월:", cashflow_target["month"].max())
print("전체 월 수:", cashflow_target["month"].nunique())

print("\n월 목록:")
print(sorted(cashflow_target["month"].unique()))

최초 월: 2025-04
최근 월: 2026-09
전체 월 수: 18

월 목록:
['2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07', '2026-08', '2026-09']


In [45]:
# 최근 12개월 데이터만 추출
cashflow_12m = cashflow_target[
    cashflow_target["month"].between("2025-10", "2026-09")
].copy()

In [46]:
print("기간:", cashflow_12m["month"].min(), "~", cashflow_12m["month"].max())
print("월 수:", cashflow_12m["month"].nunique())
print("행 수:", len(cashflow_12m))
print("고객 수:", cashflow_12m["customer_id"].nunique())

기간: 2025-10 ~ 2026-09
월 수: 12
행 수: 92928
고객 수: 7744


In [47]:
cashflow_12m.groupby("customer_id")["month"].nunique().value_counts()

month
12    7744
Name: count, dtype: int64

In [48]:
# 투자 행동 EDA 시작, investment_transfer 자체를 살펴봄
# 0원이 얼마나 많은지
print("전체 월 데이터:", len(cashflow_12m))

print(
    "투자이체 0원:",
    (cashflow_12m["investment_transfer"] == 0).sum()
)

print(
    "투자이체 > 0원:",
    (cashflow_12m["investment_transfer"] > 0).sum()
)

print(
    "투자이체 < 0원:",
    (cashflow_12m["investment_transfer"] < 0).sum()
)

전체 월 데이터: 92928
투자이체 0원: 63972
투자이체 > 0원: 28956
투자이체 < 0원: 0


In [49]:
# 비율
investment_month_ratio = (
    cashflow_12m["investment_transfer"]
    .gt(0)
    .value_counts(normalize=True)
    .mul(100)
)

investment_month_ratio

investment_transfer
False    68.840393
True     31.159607
Name: proportion, dtype: float64

In [50]:
# 금액 분포
cashflow_12m["investment_transfer"].describe()

count    9.292800e+04
mean     1.030074e+05
std      2.034791e+05
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.512241e+05
max      3.078170e+06
Name: investment_transfer, dtype: float64

In [51]:
# 실제로 투자이체가 발생한 월만 따로 보는 것
cashflow_12m.loc[
    cashflow_12m["investment_transfer"] > 0,
    "investment_transfer"
].describe()

count    2.895600e+04
mean     3.305800e+05
std      2.400957e+05
min      1.953040e+03
25%      1.719695e+05
50%      2.733089e+05
75%      4.219540e+05
max      3.078170e+06
Name: investment_transfer, dtype: float64

In [52]:
#고객마다 최근 12개월 중 몇 개월 투자했는지
investment_months = (
    cashflow_12m
    .assign(
        invested=lambda x: x["investment_transfer"] > 0
    )
    .groupby("customer_id")["invested"]
    .sum()
)

investment_months.describe()

count    7744.000000
mean        3.739153
std         5.558108
min         0.000000
25%         0.000000
50%         0.000000
75%        12.000000
max        12.000000
Name: invested, dtype: float64

In [53]:
investment_months.value_counts().sort_index()

invested
0     5331
12    2413
Name: count, dtype: int64

# 한 번 투자하는 고객은 12개월 내내 매월 투자하고, 그렇지 않은 고객은 12개월 동안 한 번도 투자하지 않는 구조

In [54]:
# 최근 12개월 투자이체 여부를 고객 단위로 생성
investment_activity = (
    cashflow_12m
    .groupby("customer_id")["investment_transfer"]
    .sum()
    .gt(0)
    .astype(int)
    .rename("has_investment_transfer")
    .reset_index()
)

# 주식 투자 여부와 결합
investment_check = target_customers[
    ["customer_id", "is_stock_investor"]
].merge(
    investment_activity,
    on="customer_id",
    how="left"
)

pd.crosstab(
    investment_check["is_stock_investor"],
    investment_check["has_investment_transfer"],
    margins=True
)

has_investment_transfer,0,1,All
is_stock_investor,,,
0,5166,0,5166
1,165,2413,2578
All,5331,2413,7744


# 즉 비주식 투자자 5,166명은 최근 12개월 동안 investment_transfer가 단 한 번도 발생하지 않았음. 반면 주식 투자자 2,578명 중 2,413명(약 93.6%)은 투자이체가 있고, 165명(약 6.4%)은 투자이체가 없음
여기서 중요한 해석은 investment_transfer > 0을 주식 투자자 판별 기준으로 쓰면 안됨. 실제 주식 투자 여부는 is_stock_investor로 이미 정의되어 있고, investment_transfer는 그 투자자들의 자금이체 행동을 나타내는 변수로 보는 게 적절함.

In [55]:
cashflow_12m[
    [
        "income_inflow",
        "savings_transfer",
        "investment_transfer"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
income_inflow,92928.0,3.578518e+06,1.926398e+06,328938.21,2.304823e+06,3210228.855,4.435992e+06,40921178.13
savings_transfer,92928.0,9.343793e+05,1.276108e+06,0.00,2.058313e+05,497388.685,1.207842e+06,27754195.95
investment_transfer,92928.0,1.030074e+05,2.034791e+05,0.00,0.000000e+00,0.000,1.512241e+05,3078169.56


In [56]:
cashflow_12m_check = cashflow_12m.merge(
    target_customers[
        ["customer_id", "is_stock_investor"]
    ],
    on="customer_id",
    how="left"
)

In [57]:

cashflow_12m_check.groupby("is_stock_investor")[
    [
        "income_inflow",
        "savings_transfer",
        "investment_transfer"
    ]
].mean().round(0)

,income_inflow,savings_transfer,investment_transfer
is_stock_investor,,,
0,3456460.0,837069.0,0.0
1,3823107.0,1129377.0,309422.0


In [58]:
print("income_inflow = 0:",
      (cashflow_12m["income_inflow"] == 0).sum())

print("savings_transfer = 0:",
      (cashflow_12m["savings_transfer"] == 0).sum())

print("investment_transfer = 0:",
      (cashflow_12m["investment_transfer"] == 0).sum())

print(
    "저축 + 투자 = 0:",
    (
        (cashflow_12m["savings_transfer"] +
         cashflow_12m["investment_transfer"]) == 0
    ).sum()
)

income_inflow = 0: 0
savings_transfer = 0: 6252
investment_transfer = 0: 63972
저축 + 투자 = 0: 6252


In [59]:
# 저축을 하지 않는 월에는 투자도 하지 않는 구조인지 확인
pd.crosstab(
    cashflow_12m["savings_transfer"] > 0,
    cashflow_12m["investment_transfer"] > 0,
    margins=True
)

investment_transfer,False,True,All
savings_transfer,,,
False,6252,0,6252
True,57720,28956,86676
All,63972,28956,92928


In [60]:
# cashflow_12m에서 이상치가 어느 정도인지 먼저 확인
def check_outliers(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = df[
        (df[column] < lower) |
        (df[column] > upper)
    ]

    print(f"[{column}]")
    print(f"Q1: {q1:,.0f}")
    print(f"Q3: {q3:,.0f}")
    print(f"IQR: {iqr:,.0f}")
    print(f"하한: {lower:,.0f}")
    print(f"상한: {upper:,.0f}")
    print(f"이상치 개수: {len(outliers):,}")
    print(f"이상치 비율: {len(outliers) / len(df) * 100:.2f}%")
    print(f"최대값: {df[column].max():,.0f}")
    print()

In [61]:
for col in [
    "income_inflow",
    "savings_transfer",
    "investment_transfer"
]:
    check_outliers(cashflow_12m, col)

[income_inflow]
Q1: 2,304,823
Q3: 4,435,992
IQR: 2,131,169
하한: -891,931
상한: 7,632,745
이상치 개수: 2,944
이상치 비율: 3.17%
최대값: 40,921,178

[savings_transfer]
Q1: 205,831
Q3: 1,207,842
IQR: 1,002,011
하한: -1,297,185
상한: 2,710,859
이상치 개수: 6,531
이상치 비율: 7.03%
최대값: 27,754,196

[investment_transfer]
Q1: 0
Q3: 151,224
IQR: 151,224
하한: -226,836
상한: 378,060
이상치 개수: 9,144
이상치 비율: 9.84%
최대값: 3,078,170



In [62]:
investment_positive = cashflow_12m[
    cashflow_12m["investment_transfer"] > 0
].copy()

check_outliers(
    investment_positive,
    "investment_transfer"
)

[investment_transfer]
Q1: 171,969
Q3: 421,954
IQR: 249,985
하한: -203,007
상한: 796,931
이상치 개수: 1,330
이상치 비율: 4.59%
최대값: 3,078,170



# 고객별 최근 12개월 투자습관 파생변수 3개를 실제로 만들어봄

In [63]:
# 고객 단위 12개월 집계
cashflow_features = (
    cashflow_12m
    .groupby("customer_id")
    .agg(
        total_income_12m=("income_inflow", "sum"),
        total_savings_12m=("savings_transfer", "sum"),
        total_investment_12m=("investment_transfer", "sum"),
        avg_monthly_investment=("investment_transfer", "mean")
    )
    .reset_index()
)

cashflow_features.head()

,customer_id,total_income_12m,total_savings_12m,total_investment_12m,avg_monthly_investment
0,37,39400644.69,1293904.66,0.00,0.00
1,72,76914839.99,10623517.68,0.00,0.00
2,78,45379136.36,4694757.31,1749651.72,145804.31
3,86,85232407.94,13860335.48,0.00,0.00
4,95,27640530.92,38634277.88,0.00,0.00


In [64]:
# 투자소득비율 생성
cashflow_features["investment_income_ratio"] = (
    cashflow_features["total_investment_12m"]
    / cashflow_features["total_income_12m"]
)

In [65]:
#투자배분비율 생성
allocation_total = (
    cashflow_features["total_investment_12m"]
    + cashflow_features["total_savings_12m"]
)

cashflow_features["investment_allocation_ratio"] = (
    cashflow_features["total_investment_12m"]
    / allocation_total.replace(0, np.nan)
)

In [66]:
# 만들어진 변수 검증
cashflow_features[
    [
        "avg_monthly_investment",
        "investment_income_ratio",
        "investment_allocation_ratio"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
avg_monthly_investment,7744.0,103007.440491,203417.130525,0.0,0.0,0.0,151481.660000,3.078170e+06
investment_income_ratio,7744.0,0.029632,0.058979,0.0,0.0,0.0,0.046576,7.799563e-01
investment_allocation_ratio,7223.0,0.098922,0.155980,0.0,0.0,0.0,0.202905,7.113278e-01


# 정리
먼저 avg_monthly_investment는 전체 7,744명 기준 평균 약 10.3만 원이지만 중앙값은 0원이야. 앞에서 확인했듯 비투자자가 많기 때문이므로 정상적인 결과야. 투자 규모를 나타내는 변수로 유지할 가치가 있어.
investment_income_ratio는 평균 0.0296, 즉 전체 고객 기준으로 최근 12개월 소득의 평균 약 3.0% 수준이 투자이체에 배분된 것으로 나타나. 다만 이것도 비투자자들의 0이 포함된 평균이야. 최대값은 약 0.78이므로 일부 고객은 소득 대비 투자비중이 상당히 높아. 이 변수 역시 투자 적극성을 표현하기 때문에 유지하는 게 좋아.
investment_allocation_ratio는 평균 약 0.099야. 즉 전체적으로는 저축+투자 자금 중 약 9.9%가 투자 쪽에 배분된 형태지만, 역시 0이 많이 포함되어 있어. 그리고 여기서 가장 중요한 게 count = 7,223이라는 점이야.

In [67]:
#결측치 확인
cashflow_features[
    [
        "avg_monthly_investment",
        "investment_income_ratio",
        "investment_allocation_ratio"
    ]
].isna().sum()

avg_monthly_investment           0
investment_income_ratio          0
investment_allocation_ratio    521
dtype: int64

In [68]:
cashflow_features_check = cashflow_features.merge(
    target_customers[
        ["customer_id", "is_stock_investor"]
    ],
    on="customer_id",
    how="left"
)

cashflow_features_check[
    cashflow_features_check["is_stock_investor"] == 1
][
    [
        "avg_monthly_investment",
        "investment_income_ratio",
        "investment_allocation_ratio"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
avg_monthly_investment,2578.0,309421.884858,245833.965424,0.000000,151909.350000,257138.815000,409233.502500,3.078170e+06
investment_income_ratio,2578.0,0.089011,0.071864,0.000000,0.046632,0.069720,0.114305,7.799563e-01
investment_allocation_ratio,2413.0,0.296109,0.120145,0.091662,0.202476,0.274634,0.365076,7.113278e-01


# cashflow EDA 결론
현재는 이렇게 확정해두자.
유지
- avg_monthly_investment → 투자 규모
- investment_income_ratio → 소득 대비 투자 적극성
- investment_allocation_ratio → 저축 대비 투자 자금배분 성향
제외
- investment_persistence_ratio → 모든 고객이 0개월 또는 12개월이라 고객 간 지속성 차이를 표현하지 못함
보조적으로 활용 가능
- has_investment_transfer → 최근 12개월 투자이체 활동 여부
그리고 이상치는 현재 삭제하지 않음. 추천 모델링 단계에서 필요하면 로그변환이나 RobustScaler 등을 검토하면 돼.

# investment_accounts EDA

In [69]:
# 구조 확인
investment_accounts_target.info()

<class 'pandas.DataFrame'>
RangeIndex: 2578 entries, 0 to 2577
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   investment_account_id  2578 non-null   int64         
 1   customer_id            2578 non-null   int64         
 2   broker_name            2578 non-null   str           
 3   account_type           2578 non-null   str           
 4   opened_at              2578 non-null   object        
 5   risk_profile           2578 non-null   str           
 6   cash_balance           2578 non-null   float64       
 7   status                 2578 non-null   str           
 8   created_at             2578 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(1), int64(2), object(1), str(4)
memory usage: 267.5+ KB


In [70]:
#전체 컬럼:
for i, col in enumerate(investment_accounts_target.columns, 1):
    print(f"{i:2}. {col}")

 1. investment_account_id
 2. customer_id
 3. broker_name
 4. account_type
 5. opened_at
 6. risk_profile
 7. cash_balance
 8. status
 9. created_at


In [71]:
investment_accounts_target.head()

,investment_account_id,customer_id,broker_name,account_type,opened_at,risk_profile,cash_balance,status,created_at
0,31,78,미래에셋증권,GENERAL,2020-05-10,BALANCED,19676414.14,ACTIVE,2026-09-13
1,39,105,KB증권,GENERAL,2024-03-18,BALANCED,4020147.29,ACTIVE,2026-09-13
2,44,116,키움증권,GENERAL,2025-04-07,AGGRESSIVE,4733112.88,ACTIVE,2026-09-13
3,60,170,한국투자증권,ISA,2024-10-17,BALANCED,2171414.93,ACTIVE,2026-09-13
4,92,249,한국투자증권,GENERAL,2025-07-06,BALANCED,294307.42,ACTIVE,2026-09-13


In [72]:
# 범주형 변수 분포부터 확인
for col in [
    "broker_name",
    "account_type",
    "risk_profile",
    "status"
]:
    print(f"\n=== {col} ===")
    print(
        investment_accounts_target[col]
        .value_counts(dropna=False)
    )


=== broker_name ===
broker_name
하나증권      337
키움증권      332
삼성증권      326
미래에셋증권    324
신한투자증권    323
한국투자증권    322
KB증권      318
NH투자증권    296
Name: count, dtype: int64

=== account_type ===
account_type
GENERAL    1795
ISA         545
CMA         238
Name: count, dtype: int64

=== risk_profile ===
risk_profile
BALANCED        2100
AGGRESSIVE       402
CONSERVATIVE      76
Name: count, dtype: int64

=== status ===
status
ACTIVE    2578
Name: count, dtype: int64


In [73]:
# cash_balance
investment_accounts_target["cash_balance"].describe()

count    2.578000e+03
mean     3.976660e+06
std      4.381136e+06
min      1.314549e+05
25%      1.374676e+06
50%      2.643918e+06
75%      4.921989e+06
max      4.767287e+07
Name: cash_balance, dtype: float64

In [74]:
# 두 위험성향 변수가 같은지
risk_compare = investment_accounts_target[
    ["customer_id", "risk_profile"]
].merge(
    target_customers[
        ["customer_id", "investment_risk_profile"]
    ],
    on="customer_id",
    how="left"
)

pd.crosstab(
    risk_compare["investment_risk_profile"],
    risk_compare["risk_profile"],
    margins=True
)

risk_profile,AGGRESSIVE,BALANCED,CONSERVATIVE,All
investment_risk_profile,,,,
AGGRESSIVE,402,0,0,402
BALANCED,0,2100,0,2100
CONSERVATIVE,0,0,76,76
All,402,2100,76,2578


In [75]:
investment_accounts_target["opened_at"] = pd.to_datetime(
    investment_accounts_target["opened_at"]
)

print(
    "가장 오래된 계좌:",
    investment_accounts_target["opened_at"].min()
)

print(
    "가장 최근 계좌:",
    investment_accounts_target["opened_at"].max()
)

investment_accounts_target["opened_at"].describe()

가장 오래된 계좌: 2019-11-10 00:00:00
가장 최근 계좌: 2026-03-17 00:00:00


count                   2578
mean     2023-01-26 13:20:59
min      2019-11-10 00:00:00
25%      2021-07-05 06:00:00
50%      2023-02-06 00:00:00
75%      2024-09-09 00:00:00
max      2026-03-17 00:00:00
Name: opened_at, dtype: object

In [76]:
account_check = investment_accounts_target[
    ["customer_id", "account_type"]
].merge(
    cashflow_features[
        [
            "customer_id",
            "avg_monthly_investment",
            "investment_income_ratio",
            "investment_allocation_ratio"
        ]
    ],
    on="customer_id",
    how="left"
)

account_check.groupby("account_type")[
    [
        "avg_monthly_investment",
        "investment_income_ratio",
        "investment_allocation_ratio"
    ]
].median().round(3)

,avg_monthly_investment,investment_income_ratio,investment_allocation_ratio
account_type,,,
CMA,255603.838,0.068,0.279
GENERAL,259110.300,0.071,0.273
ISA,251871.360,0.068,0.277


# 정리
계좌 유형에 따라 투자습관이 뚜렷하게 갈린다는 증거는 거의 없어 보이기 때문에 추천시스템 핵심 변수로 굳이 넣지 않아도 될 가능성이 높음

In [77]:
reference_date = pd.Timestamp("2026-09-30")

investment_accounts_target["investment_experience_months"] = (
    (reference_date.year - investment_accounts_target["opened_at"].dt.year) * 12
    + (reference_date.month - investment_accounts_target["opened_at"].dt.month)
)

investment_accounts_target["investment_experience_months"].describe()

count    2578.000000
mean       43.621412
std        21.896487
min         6.000000
25%        24.000000
50%        43.000000
75%        62.000000
max        82.000000
Name: investment_experience_months, dtype: float64

# customer_stock_lots EDA

In [78]:
stock_lots_target.info()

<class 'pandas.DataFrame'>
RangeIndex: 8138 entries, 0 to 8137
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   lot_id                 8138 non-null   int64         
 1   investment_account_id  8138 non-null   int64         
 2   customer_id            8138 non-null   int64         
 3   stock_code             8138 non-null   str           
 4   purchase_date          8138 non-null   object        
 5   quantity               8138 non-null   int64         
 6   purchase_price         8138 non-null   float64       
 7   purchase_fee           8138 non-null   float64       
 8   currency               8138 non-null   str           
 9   created_at             8138 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(2), int64(4), object(1), str(2)
memory usage: 707.4+ KB


In [79]:
for i, col in enumerate(stock_lots_target.columns, 1):
    print(f"{i:2}. {col}")

 1. lot_id
 2. investment_account_id
 3. customer_id
 4. stock_code
 5. purchase_date
 6. quantity
 7. purchase_price
 8. purchase_fee
 9. currency
10. created_at


In [80]:
stock_lots_target.head()

,lot_id,investment_account_id,customer_id,stock_code,purchase_date,quantity,purchase_price,purchase_fee,currency,created_at
0,100,31,78,051370,2026-07-09,105,6570.0,103.48,KRW,2026-09-13
1,101,31,78,408920,2026-05-27,257,3275.0,126.25,KRW,2026-09-13
2,102,31,78,067730,2025-08-26,42,3030.0,19.09,KRW,2026-09-13
3,127,39,105,002460,2026-07-28,1877,10710.0,3015.40,KRW,2026-09-13
4,143,44,116,051630,2025-10-22,1529,3275.0,751.12,KRW,2026-09-13


In [81]:
# 매수 데이터의 기간과 기본 구조
# 날짜형으로 변환
stock_lots_target["purchase_date"] = pd.to_datetime(
    stock_lots_target["purchase_date"]
)

print("최초 매수일:", stock_lots_target["purchase_date"].min())
print("최근 매수일:", stock_lots_target["purchase_date"].max())
print("전체 lot 수:", len(stock_lots_target))
print("고객 수:", stock_lots_target["customer_id"].nunique())
print("종목 수:", stock_lots_target["stock_code"].nunique())

최초 매수일: 2024-08-28 00:00:00
최근 매수일: 2026-09-11 00:00:00
전체 lot 수: 8138
고객 수: 2578
종목 수: 600


In [82]:
# currency도 확인
stock_lots_target["currency"].value_counts(dropna=False)

currency
KRW    8138
Name: count, dtype: int64

In [83]:
# 고객당 lot 수
lots_per_customer = (
    stock_lots_target
    .groupby("customer_id")
    .size()
)

lots_per_customer.describe()

count    2578.000000
mean        3.156711
std         1.448734
min         1.000000
25%         2.000000
50%         3.000000
75%         4.000000
max         6.000000
dtype: float64

In [84]:
# 분포
lots_per_customer.value_counts().sort_index()

1    364
2    555
3    667
4    489
5    311
6    192
Name: count, dtype: int64

In [85]:
# 최근 12개월 이전 매수 lot이 얼마나 있는지
cutoff_date = pd.Timestamp("2025-10-01")

print(
    "최근 12개월 이전 lot:",
    (stock_lots_target["purchase_date"] < cutoff_date).sum()
)

print(
    "최근 12개월 lot:",
    (stock_lots_target["purchase_date"] >= cutoff_date).sum()
)

최근 12개월 이전 lot: 2296
최근 12개월 lot: 5842


In [86]:
# 고객 기준 
lot_period_check = (
    stock_lots_target
    .assign(
        recent_12m=stock_lots_target["purchase_date"] >= cutoff_date
    )
    .groupby("customer_id")
    .agg(
        total_lots=("lot_id", "count"),
        recent_lots=("recent_12m", "sum")
    )
)

lot_period_check["old_lots"] = (
    lot_period_check["total_lots"]
    - lot_period_check["recent_lots"]
)

print("전체 투자자:", len(lot_period_check))

print(
    "최근 12개월 매수 lot이 있는 고객:",
    (lot_period_check["recent_lots"] > 0).sum()
)

print(
    "12개월 이전 lot을 보유한 고객:",
    (lot_period_check["old_lots"] > 0).sum()
)

print(
    "최근 12개월 lot만 있는 고객:",
    (lot_period_check["old_lots"] == 0).sum()
)

전체 투자자: 2578
최근 12개월 매수 lot이 있는 고객: 2399
12개월 이전 lot을 보유한 고객: 1471
최근 12개월 lot만 있는 고객: 1107


In [87]:
# 같은 고객이 같은 날짜에 같은 종목을 여러 lot으로 가지고 있는 경우가 있는지
duplicate_purchase = (
    stock_lots_target
    .groupby(
        ["customer_id", "stock_code", "purchase_date"]
    )
    .size()
    .reset_index(name="lot_count")
)

print(
    "고객-종목-매수일 조합 수:",
    len(duplicate_purchase)
)

print(
    "같은 고객·종목·날짜에 여러 lot이 있는 조합:",
    (duplicate_purchase["lot_count"] > 1).sum()
)

print(
    "최대 lot 수:",
    duplicate_purchase["lot_count"].max()
)

고객-종목-매수일 조합 수: 8138
같은 고객·종목·날짜에 여러 lot이 있는 조합: 0
최대 lot 수: 1


In [88]:
duplicate_purchase[
    duplicate_purchase["lot_count"] > 1
].head(10)

,customer_id,stock_code,purchase_date,lot_count


In [89]:
stock_lots_target["purchase_amount"] = (
    stock_lots_target["quantity"]
    * stock_lots_target["purchase_price"]
)

stock_lots_target["purchase_amount"].describe()

count    8.138000e+03
mean     5.641952e+06
std      1.139239e+07
min      3.050000e+03
25%      5.070125e+05
50%      1.663795e+06
75%      5.611785e+06
max      1.634502e+08
Name: purchase_amount, dtype: float64

In [90]:
# 전체 기간 고객별 취득금액
total_purchase = (
    stock_lots_target
    .groupby("customer_id")
    .agg(
        total_purchase_count=("lot_id", "count"),
        total_purchase_amount=("purchase_amount", "sum")
    )
)

# 최근 12개월
stock_lots_12m = stock_lots_target[
    stock_lots_target["purchase_date"] >= "2025-10-01"
].copy()

recent_purchase = (
    stock_lots_12m
    .groupby("customer_id")
    .agg(
        purchase_count_12m=("lot_id", "count"),
        purchase_amount_12m=("purchase_amount", "sum")
    )
)

# 전체 투자자 2,578명을 기준으로 결합
stock_lot_features = (
    investment_accounts_target[["customer_id"]]
    .merge(total_purchase, on="customer_id", how="left")
    .merge(recent_purchase, on="customer_id", how="left")
)

# 최근 12개월 매수 기록이 없는 고객은 0
stock_lot_features[
    ["purchase_count_12m", "purchase_amount_12m"]
] = stock_lot_features[
    ["purchase_count_12m", "purchase_amount_12m"]
].fillna(0)

In [91]:
stock_lot_features[
    [
        "total_purchase_count",
        "total_purchase_amount",
        "purchase_count_12m",
        "purchase_amount_12m"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
total_purchase_count,2578.0,3.156711e+00,1.448734e+00,1.0,2.0,3.0,4.00,6.0
total_purchase_amount,2578.0,1.781001e+07,2.616007e+07,669000.0,2508329.0,7275792.5,22546976.25,326359255.0
purchase_count_12m,2578.0,2.266098e+00,1.346096e+00,0.0,1.0,2.0,3.00,6.0
purchase_amount_12m,2578.0,1.305980e+07,2.227673e+07,0.0,1307885.0,4381170.0,15545760.00,271046890.0


In [92]:
lot_cashflow_check = stock_lot_features.merge(
    cashflow_features[
        [
            "customer_id",
            "total_income_12m",
            "total_investment_12m"
        ]
    ],
    on="customer_id",
    how="left"
)

lot_cashflow_check[
    [
        "purchase_amount_12m",
        "total_income_12m",
        "total_investment_12m"
    ]
].corr().round(3)

,purchase_amount_12m,total_income_12m,total_investment_12m
purchase_amount_12m,1.000,0.066,0.070
total_income_12m,0.066,1.000,0.481
total_investment_12m,0.070,0.481,1.000


In [93]:
# 최근 12개월 매수 종목 다양성
stock_lots_12m.groupby("customer_id")["stock_code"].nunique().describe()

count    2399.000000
mean        2.435181
std         1.239076
min         1.000000
25%         1.000000
50%         2.000000
75%         3.000000
max         6.000000
Name: stock_code, dtype: float64

In [94]:
# 분포
stock_lots_12m.groupby("customer_id")["stock_code"].nunique().value_counts().sort_index()

stock_code
1    638
2    736
3    573
4    293
5    113
6     46
Name: count, dtype: int64

In [95]:
recent_stock_diversity = (
    stock_lots_12m
    .groupby("customer_id")
    .agg(
        purchase_count_12m=("lot_id", "count"),
        unique_stocks_12m=("stock_code", "nunique")
    )
)

print(
    "상관계수:",
    recent_stock_diversity[
        ["purchase_count_12m", "unique_stocks_12m"]
    ].corr().iloc[0, 1]
)

print(
    "매수 횟수 = 매수 종목 수:",
    (
        recent_stock_diversity["purchase_count_12m"]
        == recent_stock_diversity["unique_stocks_12m"]
    ).sum()
)

print(
    "같은 종목 반복 매수:",
    (
        recent_stock_diversity["purchase_count_12m"]
        > recent_stock_diversity["unique_stocks_12m"]
    ).sum()
)

상관계수: 1.0
매수 횟수 = 매수 종목 수: 2399
같은 종목 반복 매수: 0


# customer_stock_positions EDA

In [96]:
stock_positions_target.info()

<class 'pandas.DataFrame'>
RangeIndex: 8138 entries, 0 to 8137
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   customer_id                  8138 non-null   int64  
 1   investment_account_id        8138 non-null   int64  
 2   stock_code                   8138 non-null   str    
 3   company_name                 8138 non-null   str    
 4   market                       8138 non-null   str    
 5   industry                     8138 non-null   str    
 6   purchase_date                8138 non-null   object 
 7   purchase_price_actual_close  8138 non-null   float64
 8   quantity                     8138 non-null   int64  
 9   latest_actual_close          8138 non-null   float64
 10  current_value                8138 non-null   float64
 11  unrealized_return_rate       8138 non-null   float64
 12  volatility_annualized        8138 non-null   float64
 13  volatility_bucket            

In [97]:
for i, col in enumerate(stock_positions_target.columns, 1):
    print(f"{i:2}. {col}")

 1. customer_id
 2. investment_account_id
 3. stock_code
 4. company_name
 5. market
 6. industry
 7. purchase_date
 8. purchase_price_actual_close
 9. quantity
10. latest_actual_close
11. current_value
12. unrealized_return_rate
13. volatility_annualized
14. volatility_bucket
15. momentum_6m
16. reference_price_data


In [98]:
stock_positions_target.head()

,customer_id,investment_account_id,stock_code,company_name,market,industry,purchase_date,purchase_price_actual_close,quantity,latest_actual_close,current_value,unrealized_return_rate,volatility_annualized,volatility_bucket,momentum_6m,reference_price_data
0,78,31,051370,인터플렉스,코스닥,전자부품 제조업,2026-07-09,6570.0,105,6120.0,642600.0,-0.068493,0.506949,MEDIUM,-0.549005,1
1,78,31,408920,메쎄이상,코스닥,기타 사업지원 서비스업,2026-05-27,3275.0,257,2970.0,763290.0,-0.093130,0.402885,LOW,-0.063091,1
2,78,31,067730,로지시스,코스닥,"컴퓨터 프로그래밍, 시스템 통합 및 관리업",2025-08-26,3030.0,42,2240.0,94080.0,-0.260726,0.478921,MEDIUM,-0.072464,1
3,105,39,002460,HS화성,유가,건물 건설업,2026-07-28,10710.0,1877,10830.0,20327910.0,0.011204,0.360079,LOW,-0.150588,1
4,116,44,051630,진양화학,유가,플라스틱제품 제조업,2025-10-22,3275.0,1529,1362.0,2082498.0,-0.584122,0.886356,HIGH,-0.397345,1


In [99]:
# 1. 현재 보유 종목 수
position_features = (
    stock_positions_target
    .groupby("customer_id")
    .agg(
        unique_stocks=("stock_code", "nunique"),
        unique_industries=("industry", "nunique"),
        total_current_value=("current_value", "sum")
    )
    .reset_index()
)

position_features[
    [
        "unique_stocks",
        "unique_industries",
        "total_current_value"
    ]
].describe().T

# unique_stocks → 현재 보유 종목 다양성
# unique_industries → 현재 보유 산업 다양성
# total_current_value → 현재 주식자산 평가금액

,count,mean,std,min,25%,50%,75%,max
unique_stocks,2578.0,3.156711e+00,1.448734e+00,1.0,2.0,3.0,4.0,6.000000e+00
unique_industries,2578.0,3.039178e+00,1.387208e+00,1.0,2.0,3.0,4.0,6.000000e+00
total_current_value,2578.0,1.744069e+07,3.469985e+07,45240.0,2258531.0,6608692.5,20854707.0,1.170823e+09


In [100]:
# 2. 종목 다양성과 산업 다양성이 중복인지 확인
print(
    position_features[
        ["unique_stocks", "unique_industries"]
    ].corr()
)

print(
    "종목 수 = 산업 수:",
    (
        position_features["unique_stocks"]
        == position_features["unique_industries"]
    ).sum()
)

print(
    "종목 수 > 산업 수:",
    (
        position_features["unique_stocks"]
        > position_features["unique_industries"]
    ).sum()
)

                   unique_stocks  unique_industries
unique_stocks           1.000000           0.970879
unique_industries       0.970879           1.000000
종목 수 = 산업 수: 2296
종목 수 > 산업 수: 282


In [101]:
# 먼저 positions 날짜도 변환
stock_positions_target["purchase_date"] = pd.to_datetime(
    stock_positions_target["purchase_date"]
)

print(stock_lots_target["purchase_date"].dtype)
print(stock_positions_target["purchase_date"].dtype)

datetime64[s]
datetime64[s]


In [102]:
# 3. stock_lots와 stock_positions 관계도 확인
position_lot_check = stock_positions_target.merge(
    stock_lots_target[
        [
            "customer_id",
            "stock_code",
            "purchase_date",
            "quantity"
        ]
    ],
    on=[
        "customer_id",
        "stock_code",
        "purchase_date",
        "quantity"
    ],

    how="outer",
    indicator=True
)

position_lot_check["_merge"].value_counts()

_merge
both          8138
left_only        0
right_only       0
Name: count, dtype: int64

In [103]:
stock_positions_target["volatility_annualized"].describe()

count    8138.000000
mean        0.557913
std         0.372059
min         0.028930
25%         0.383662
50%         0.508915
75%         0.665585
max         6.423851
Name: volatility_annualized, dtype: float64

In [104]:
stock_positions_target["volatility_bucket"].value_counts()

volatility_bucket
LOW       3538
MEDIUM    3367
HIGH      1233
Name: count, dtype: int64

In [105]:
# 고객별 평균 변동성
customer_volatility = (
    stock_positions_target
    .groupby("customer_id")
    .agg(
        avg_stock_volatility=("volatility_annualized", "mean")
    )
    .reset_index()
)

customer_volatility["avg_stock_volatility"].describe()

count    2578.000000
mean        0.562175
std         0.293429
min         0.028930
25%         0.429637
50%         0.498689
75%         0.587146
max         6.391114
Name: avg_stock_volatility, dtype: float64

In [106]:
# 현재가치 가중 평균 변동성을 하나 만들어 비교
# 각 포지션의 변동성 × 현재가치
stock_positions_target["weighted_volatility"] = (
    stock_positions_target["volatility_annualized"]
    * stock_positions_target["current_value"]
)

weighted_volatility = (
    stock_positions_target
    .groupby("customer_id")
    .agg(
        weighted_vol_sum=("weighted_volatility", "sum"),
        total_current_value=("current_value", "sum")
    )
)

weighted_volatility["weighted_avg_volatility"] = (
    weighted_volatility["weighted_vol_sum"]
    / weighted_volatility["total_current_value"]
)

weighted_volatility["weighted_avg_volatility"].describe()

count    2578.000000
mean        0.573346
std         0.398557
min         0.028930
25%         0.418192
50%         0.498857
75%         0.609028
max         6.391114
Name: weighted_avg_volatility, dtype: float64

In [107]:
# 단순 평균과 얼마나 다른 정보인지 확인
volatility_compare = customer_volatility.merge(
    weighted_volatility[
        ["weighted_avg_volatility"]
    ],
    left_on="customer_id",
    right_index=True
)

volatility_compare[
    ["avg_stock_volatility", "weighted_avg_volatility"]
].corr()

,avg_stock_volatility,weighted_avg_volatility
avg_stock_volatility,1.000000,0.915325
weighted_avg_volatility,0.915325,1.000000


In [108]:
stock_positions_target[
    [
        "unrealized_return_rate",
        "momentum_6m"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
unrealized_return_rate,8138.0,-0.019122,0.608653,-0.997636,-0.275900,-0.082215,0.073091,12.017479
momentum_6m,8138.0,-0.131072,0.410237,-0.973913,-0.339181,-0.164447,-0.014356,5.913043


In [109]:
stock_positions_target[
    [
        "unrealized_return_rate",
        "momentum_6m",
        "volatility_annualized"
    ]
].corr().round(3)

,unrealized_return_rate,momentum_6m,volatility_annualized
unrealized_return_rate,1.000,0.63,0.334
momentum_6m,0.630,1.00,0.430
volatility_annualized,0.334,0.43,1.000


# customer_debt_summary EDA

In [110]:
debt_summary_target.info()

for i, col in enumerate(debt_summary_target.columns, 1):
    print(f"{i:2}. {col}")

<class 'pandas.DataFrame'>
Index: 7744 entries, 0 to 7746
Data columns (total 23 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   customer_id                         7744 non-null   int64  
 1   is_home_owner                       7744 non-null   int64  
 2   home_value                          7744 non-null   float64
 3   mortgage_balance                    7744 non-null   float64
 4   credit_line_balance                 7744 non-null   float64
 5   personal_loan_balance               7744 non-null   float64
 6   total_financial_debt                7744 non-null   float64
 7   monthly_debt_service                7744 non-null   float64
 8   annual_debt_service                 7744 non-null   float64
 9   dsr_income                          7744 non-null   float64
 10  debt_service_to_disposable          7744 non-null   float64
 11  debt_to_income                      7744 non-null   float64

In [111]:
debt_summary_target.head()

,customer_id,is_home_owner,home_value,mortgage_balance,credit_line_balance,personal_loan_balance,total_financial_debt,monthly_debt_service,annual_debt_service,dsr_income,...,mortgage_ltv_current,monthly_private_education_cost,housing_education_fixed_burden,housing_education_burden_ratio,high_debt_service_flag,high_fixed_burden_flag,calibration_total_debt_avg_age,calibration_financial_debt_avg_age,calibration_secured_debt_avg_age,calibration_credit_debt_avg_age
0,37,1,3.280000e+08,1.410610e+08,13321421.20,0.0,1.543824e+08,931731.72,11180780.63,0.241444,...,0.430064,0.0,931731.72,0.300205,0,0,95480000,82720000,68980000,10390000
1,72,0,0.000000e+00,0.000000e+00,0.00,0.0,0.000000e+00,0.00,0.00,0.000000,...,0.000000,0.0,0.00,0.000000,0,0,95480000,82720000,68980000,10390000
2,78,0,0.000000e+00,0.000000e+00,0.00,0.0,0.000000e+00,0.00,0.00,0.000000,...,0.000000,0.0,0.00,0.000000,0,0,95480000,82720000,68980000,10390000
3,86,1,1.810000e+09,7.280482e+08,58629458.26,0.0,7.866777e+08,4053347.45,48640169.46,0.292983,...,0.402237,0.0,4053347.45,0.372406,0,0,95480000,82720000,68980000,10390000
4,95,0,0.000000e+00,0.000000e+00,0.00,0.0,0.000000e+00,0.00,0.00,0.000000,...,0.000000,0.0,0.00,0.000000,0,0,95480000,82720000,68980000,10390000


In [112]:
debt_summary_target["debt_to_income"].describe()

count    7744.000000
mean        0.746020
std         1.402059
min         0.000000
25%         0.000000
50%         0.000000
75%         0.207628
max         8.176418
Name: debt_to_income, dtype: float64

In [113]:
# 주식 투자자와 비투자자의 차이
debt_check = debt_summary_target[
    [
        "customer_id",
        "dsr_income",
        "debt_service_to_disposable",
        "debt_to_income",
        "debt_to_asset"
    ]
].merge(
    target_customers[
        ["customer_id", "is_stock_investor"]
    ],
    on="customer_id",
    how="left"
)

debt_check.groupby("is_stock_investor")[
    [
        "dsr_income",
        "debt_service_to_disposable",
        "debt_to_income",
        "debt_to_asset"
    ]
].agg(["mean", "median"]).round(3)

dsr_income        debt_service_to_disposable         \
                        mean median                       mean median   
is_stock_investor                                                       
0                      0.059    0.0                      0.072    0.0   
1                      0.074    0.0                      0.091    0.0   

                  debt_to_income        debt_to_asset         
                            mean median          mean median  
is_stock_investor                                             
0                          0.658    0.0         0.119    0.0  
1                          0.922    0.0         0.155    0.0

In [114]:
# 네개의 중복 확인
debt_check[
    [
        "dsr_income",
        "debt_service_to_disposable",
        "debt_to_income",
        "debt_to_asset"
    ]
].corr().round(3)

,dsr_income,debt_service_to_disposable,debt_to_income,debt_to_asset
dsr_income,1.000,0.999,0.923,0.856
debt_service_to_disposable,0.999,1.000,0.926,0.860
debt_to_income,0.923,0.926,1.000,0.874
debt_to_asset,0.856,0.860,0.874,1.000


In [ ]:
# 후보들을 고객 1명 = 1행 형태로 합치는 단계

In [115]:
investment_features = target_customers[
    [
        "customer_id",
        "is_stock_investor",
        "investment_propensity",
        "investment_risk_profile"
    ]
].copy()

In [116]:
investment_features = investment_features.merge(
    cashflow_features[
        [
            "customer_id",
            "avg_monthly_investment",
            "investment_income_ratio",
            "investment_allocation_ratio"
        ]
    ],
    on="customer_id",
    how="left"
)

In [117]:
investment_features = investment_features.merge(
    investment_accounts_target[
        ["customer_id", "investment_experience_months"]
    ],
    on="customer_id",
    how="left"
)

In [118]:
investment_features = investment_features.merge(
    stock_lot_features[
        [
            "customer_id",
            "purchase_count_12m",
            "purchase_amount_12m"
        ]
    ],
    on="customer_id",
    how="left"
)

In [119]:
position_final = (
    position_features[
        ["customer_id", "unique_stocks"]
    ]
    .merge(
        weighted_volatility[
            ["weighted_avg_volatility"]
        ],
        left_on="customer_id",
        right_index=True,
        how="left"
    )
)

investment_features = investment_features.merge(
    position_final,
    on="customer_id",
    how="left"
)

In [120]:
investment_features = investment_features.merge(
    debt_summary_target[
        ["customer_id", "debt_to_income"]
    ],
    on="customer_id",
    how="left"
)

In [121]:
print("shape:", investment_features.shape)
print("고객 수:", investment_features["customer_id"].nunique())

investment_features.info()

shape: (7744, 13)
고객 수: 7744
<class 'pandas.DataFrame'>
RangeIndex: 7744 entries, 0 to 7743
Data columns (total 13 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   customer_id                   7744 non-null   int64  
 1   is_stock_investor             7744 non-null   int64  
 2   investment_propensity         7744 non-null   float64
 3   investment_risk_profile       7744 non-null   str    
 4   avg_monthly_investment        7744 non-null   float64
 5   investment_income_ratio       7744 non-null   float64
 6   investment_allocation_ratio   7223 non-null   float64
 7   investment_experience_months  2578 non-null   float64
 8   purchase_count_12m            2578 non-null   float64
 9   purchase_amount_12m           2578 non-null   float64
 10  unique_stocks                 2578 non-null   float64
 11  weighted_avg_volatility       2578 non-null   float64
 12  debt_to_income                7744 non-null 

In [122]:
investment_features.isna().sum()

customer_id                        0
is_stock_investor                  0
investment_propensity              0
investment_risk_profile            0
avg_monthly_investment             0
investment_income_ratio            0
investment_allocation_ratio      521
investment_experience_months    5166
purchase_count_12m              5166
purchase_amount_12m             5166
unique_stocks                   5166
weighted_avg_volatility         5166
debt_to_income                     0
dtype: int64

In [124]:
# 투자자 데이터셋 만들기 + 결측치 확인
investor_features = investment_features[
    investment_features["is_stock_investor"] == 1
].copy()

print("투자자 수:", len(investor_features))
print()
print(investor_features.isna().sum())

투자자 수: 2578

customer_id                       0
is_stock_investor                 0
investment_propensity             0
investment_risk_profile           0
avg_monthly_investment            0
investment_income_ratio           0
investment_allocation_ratio     165
investment_experience_months      0
purchase_count_12m                0
purchase_amount_12m               0
unique_stocks                     0
weighted_avg_volatility           0
debt_to_income                    0
dtype: int64


In [126]:
# 전체/투자자/비투자자 수 확인
print("전체 분석 고객:", len(investment_features))
print("투자자:", len(investor_features))
print("비투자자:", (investment_features["is_stock_investor"] == 0).sum())

전체 분석 고객: 7744
투자자: 2578
비투자자: 5166


In [127]:
# 싱관관계 코드 실행
candidate_cols = [
    "investment_propensity",
    "avg_monthly_investment",
    "investment_income_ratio",
    "investment_allocation_ratio",
    "investment_experience_months",
    "purchase_count_12m",
    "purchase_amount_12m",
    "unique_stocks",
    "weighted_avg_volatility",
    "debt_to_income"
]

corr = investor_features[candidate_cols].corr()

corr.round(3)

,investment_propensity,avg_monthly_investment,investment_income_ratio,investment_allocation_ratio,investment_experience_months,purchase_count_12m,purchase_amount_12m,unique_stocks,weighted_avg_volatility,debt_to_income
investment_propensity,1.000,0.601,0.439,-0.210,-0.014,0.008,0.062,-0.009,0.382,0.188
avg_monthly_investment,0.601,1.000,0.637,-0.528,-0.012,0.023,0.070,0.012,0.319,-0.015
investment_income_ratio,0.439,0.637,1.000,-0.370,0.006,0.028,0.037,0.024,0.201,-0.073
investment_allocation_ratio,-0.210,-0.528,-0.370,1.000,-0.023,0.033,-0.047,0.031,-0.101,0.376
investment_experience_months,-0.014,-0.012,0.006,-0.023,1.000,-0.164,-0.073,0.028,0.007,-0.020
purchase_count_12m,0.008,0.023,0.028,0.033,-0.164,1.000,0.142,0.756,-0.029,0.003
purchase_amount_12m,0.062,0.070,0.037,-0.047,-0.073,0.142,1.000,-0.000,0.003,-0.005
unique_stocks,-0.009,0.012,0.024,0.031,0.028,0.756,-0.000,1.000,-0.006,-0.012
weighted_avg_volatility,0.382,0.319,0.201,-0.101,0.007,-0.029,0.003,-0.006,1.000,0.055
debt_to_income,0.188,-0.015,-0.073,0.376,-0.020,0.003,-0.005,-0.012,0.055,1.000


In [128]:
# 변수 설정
final_investment_features = [
    "investment_income_ratio",          # 투자 적극성
    "investment_experience_months",     # 투자 경험
    "purchase_count_12m",               # 매수 활동성
    "unique_stocks",                    # 분산 투자 성향
    "weighted_avg_volatility"           # 위험 감수 성향
]

In [129]:
# 포트폴리오 집중도 
# 고객별 전체 현재 주식자산
stock_positions_target["total_portfolio_value"] = (
    stock_positions_target
    .groupby("customer_id")["current_value"]
    .transform("sum")
)

# 각 종목이 포트폴리오에서 차지하는 비중
stock_positions_target["portfolio_weight"] = (
    stock_positions_target["current_value"]
    / stock_positions_target["total_portfolio_value"]
)

# 비중 제곱
stock_positions_target["weight_squared"] = (
    stock_positions_target["portfolio_weight"] ** 2
)

# 고객별 HHI
portfolio_concentration = (
    stock_positions_target
    .groupby("customer_id")
    .agg(
        portfolio_hhi=("weight_squared", "sum"),
        max_stock_weight=("portfolio_weight", "max")
    )
    .reset_index()
)

portfolio_concentration[
    ["portfolio_hhi", "max_stock_weight"]
].describe().T

,count,mean,std,min,25%,50%,75%,max
portfolio_hhi,2578.0,0.545605,0.245542,0.175187,0.354792,0.488475,0.696771,1.0
max_stock_weight,2578.0,0.640472,0.218461,0.214503,0.466641,0.597163,0.818242,1.0


In [130]:
concentration_check = position_features[
    ["customer_id", "unique_stocks"]
].merge(
    portfolio_concentration[
        ["customer_id", "portfolio_hhi", "max_stock_weight"]
    ],
    on="customer_id",
    how="left"
)

concentration_check[
    ["unique_stocks", "portfolio_hhi", "max_stock_weight"]
].corr().round(3)

,unique_stocks,portfolio_hhi,max_stock_weight
unique_stocks,1.000,-0.827,-0.781
portfolio_hhi,-0.827,1.000,0.979
max_stock_weight,-0.781,0.979,1.000


# 핵심 투자습관 지표
| 습관 | 최종 지표 | 의미 |
|---|---|---|
| **투자 적극성** | `investment_income_ratio` | 소득 대비 얼마나 투자하는가 |
| **투자 경험** | `investment_experience_months` | 얼마나 오래 투자해왔는가 |
| **매수 활동성** | `purchase_count_12m` | 최근 12개월 얼마나 활발하게 매수하는가 |
| **집중/분산 성향** | `portfolio_hhi` | 특정 종목에 집중하는가, 분산하는가 |
| **위험 감수 성향** | `weighted_avg_volatility` | 실제 자금을 어느 정도 위험한 종목에 배분하는가 |

In [131]:
investor_features = investor_features.merge(
    portfolio_concentration[
        ["customer_id", "portfolio_hhi", "max_stock_weight"]
    ],
    on="customer_id",
    how="left"
)

print(investor_features.shape)

investor_features[
    ["portfolio_hhi", "max_stock_weight"]
].isna().sum()

(2578, 15)


portfolio_hhi       0
max_stock_weight    0
dtype: int64

In [132]:
final_habit_cols = [
    "investment_income_ratio",          # 투자 적극성
    "investment_experience_months",     # 투자 경험
    "purchase_count_12m",               # 매수 활동성
    "portfolio_hhi",                    # 집중/분산 성향
    "weighted_avg_volatility"           # 위험 감수 성향
]

In [133]:
investor_features[
    final_habit_cols
].describe().T

,count,mean,std,min,25%,50%,75%,max
investment_income_ratio,2578.0,0.089011,0.071864,0.000000,0.046632,0.069720,0.114305,0.779956
investment_experience_months,2578.0,43.621412,21.896487,6.000000,24.000000,43.000000,62.000000,82.000000
purchase_count_12m,2578.0,2.266098,1.346096,0.000000,1.000000,2.000000,3.000000,6.000000
portfolio_hhi,2578.0,0.545605,0.245542,0.175187,0.354792,0.488475,0.696771,1.000000
weighted_avg_volatility,2578.0,0.573346,0.398557,0.028930,0.418192,0.498857,0.609028,6.391114


In [134]:
final_corr = investor_features[
    final_habit_cols
].corr()

final_corr.round(3)

,investment_income_ratio,investment_experience_months,purchase_count_12m,portfolio_hhi,weighted_avg_volatility
investment_income_ratio,1.000,0.006,0.028,-0.015,0.201
investment_experience_months,0.006,1.000,-0.164,-0.013,0.007
purchase_count_12m,0.028,-0.164,1.000,-0.643,-0.029
portfolio_hhi,-0.015,-0.013,-0.643,1.000,0.067
weighted_avg_volatility,0.201,0.007,-0.029,0.067,1.000


In [135]:
optional_amount_col = "purchase_amount_12m"